# Three checkout designs, ten shoppers: Cochran's Q test

**Cochran's Q Test** · a worked example from the StatNex teaching guide

**The story.** Ten shoppers each tried to buy something using three checkout page designs. For each design we recorded whether they finished the purchase.

**The question.** Do the three checkout designs lead to different completion rates?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `06_nonparametric/cochran_q.csv` (10 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Shopper,Design A,Design B,Design C
1.0,1.0,1.0,1.0
2.0,1.0,1.0,0.0
3.0,1.0,1.0,0.0
4.0,1.0,0.0,0.0
5.0,1.0,1.0,0.0
6.0,1.0,0.0,1.0
7.0,0.0,0.0,0.0
8.0,1.0,1.0,0.0
9.0,0.0,0.0,0.0
10.0,1.0,0.0,0.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.contingency_tables import cochrans_q

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per case, one column per condition, every cell 0 or 1. The test needs every
# case in every condition, so a blank or text cell stops it rather than being dropped.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
levels = ["Design A", "Design B", "Design C"]
values = data[levels].apply(pd.to_numeric, errors="coerce")
if values.isna().any().any() or not values.isin([0, 1]).all().all():
    raise ValueError("Cochran's Q needs a 0 or a 1 in every cell.")

## Frequencies

In [ ]:
ones = values.sum()
counts = pd.DataFrame([len(values) - ones, ones], index=["value 0", "value 1"])
print(counts)

## Cochran's Q test

In [ ]:
# McNemar's test extended to k matched conditions: each case is tried under every
# condition, and the test asks whether the share of 1s differs between conditions by
# more than chance. A case that answered the same way in every condition carries no
# information about the conditions. Q is compared with chi-square on k - 1 df, a
# large-sample approximation: with few cases that changed their answer, treat P as
# rough.
result = cochrans_q(values.to_numpy(int))
print(f"\nQ = {result.statistic:.6f}, df = {result.df}, P = {result.pvalue:.6g}")

## What the result says

Cochran's Q test compares yes or no results when the same people try every option. Here each of the 10 shoppers tried all three checkout designs.

The frequency table shows that 8 shoppers finished the purchase with Design A, 5 with Design B and 2 with Design C. The test statistic is a chi-square of 7.71 with 2 degrees of freedom (the number of designs minus one). Its P value is 0.021. The P value is the chance of differences this large if all three designs worked equally well. It is small, so the completion rates probably differ.

Treat this as a lead, not a verdict. Ten shoppers are few, and the test does not say which designs differ. Design A looks best, so test it again with more shoppers.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.